# Sesión 8 · Notebook de clase · Filtrar, ordenar y agrupar

**Fundamentos de Ciencia de Datos e IA (CDIAG1003)** · Semana 4 · Unidad 3: tema 3.1 (filtrado, orden, `groupby`, `pivot_table`, `crosstab`).

Hasta hoy la tabla era el objeto de trabajo: cargarla, seleccionar, limpiar. Desde hoy la tabla es la **materia prima de las respuestas**: cada pregunta de negocio de la sesión 3 («¿qué ciudad vende más?», «¿cuál es el ticket promedio por categoría?») es una operación de agrupar y resumir. Los datos son `datos/ventas_tienda.csv`: la tabla limpia de la sesión 5 (y la que recuperaron al limpiar `ventas_sucio.csv` en la sesión 7).

**Cómo trabajar con este notebook**
- Ejecute las celdas en orden. En los ejercicios **Su turno**, reemplace los `...`; la celda `assert` siguiente comprueba el resultado.
- Después de cada ejercicio hay una celda «Para continuar»: ejecútela siempre.
- Las salidas de la versión resuelta están en pandas 3 (texto con tipo `str`; en pandas 2 aparece `object`).

In [2]:
import numpy as np
import pandas as pd

v = pd.read_csv("datos/ventas_tienda.csv", parse_dates=["fecha"])
v["ingreso"] = v["cantidad"] * v["precio_unitario"]
v["mes"] = v["fecha"].dt.month
print(v.shape, "| ingreso total:", round(v["ingreso"].sum(), 1))
v.head()

(200, 8) | ingreso total: 7424.8


,fecha,producto,categoria,cantidad,precio_unitario,ciudad,ingreso,mes
0,2026-01-02,Sartén,Hogar,1,18.0,Quito,18.0,1
1,2026-01-02,Mouse,Electrónica,1,15.0,Machala,15.0,1
2,2026-01-03,Galletas,Alimentos,2,1.8,Quito,3.6,1
3,2026-01-03,Cuaderno,Papelería,2,2.5,Guayaquil,5.0,1
4,2026-01-04,Cargador,Electrónica,3,12.5,Guayaquil,37.5,1


## 1. Filtrar y ordenar

Filtrar ya lo conoce: una condición booleana entre corchetes. Hoy se añaden tres atajos y la ordenación:

| Quiero… | Código |
| :-- | :-- |
| filas cuyo valor está en una lista | `v[v["ciudad"].isin(["Cuenca", "Manta"])]` |
| filas con un valor en un rango (incluidos los extremos) | `v[v["precio_unitario"].between(10, 20)]` |
| filas cuyo texto contiene algo | `v[v["producto"].str.contains("Caf")]` |
| lo mismo, escrito como una expresión | `v.query("ciudad == 'Quito' and cantidad >= 5")` |
| ordenar por una columna | `v.sort_values("ingreso", ascending=False)` |
| ordenar por varias (una ascendente y otra no) | `v.sort_values(["ciudad", "ingreso"], ascending=[True, False])` |
| las n filas más grandes o más pequeñas | `v.nlargest(5, "ingreso")`, `v.nsmallest(5, "ingreso")` |

In [3]:
v.query("ciudad == 'Quito' and cantidad >= 5").shape   # las 16 ventas de la sesión 5

(16, 8)

In [4]:
# Ordenar: las ventas más grandes del trimestre. El índice NO cambia: sigue siendo la etiqueta original.
v.sort_values("ingreso", ascending=False).head(5)

,fecha,producto,categoria,cantidad,precio_unitario,ciudad,ingreso,mes
57,2026-01-28,Licuadora,Hogar,6,45.0,Guayaquil,270.0,1
188,2026-03-28,Audífonos,Electrónica,10,25.0,Quito,250.0,3
72,2026-02-07,Licuadora,Hogar,5,45.0,Guayaquil,225.0,2
114,2026-02-23,Licuadora,Hogar,5,45.0,Quito,225.0,2
140,2026-03-08,Audífonos,Electrónica,8,25.0,Manta,200.0,3


In [5]:
# Ordenar por dos criterios: ciudad A-Z y, dentro de cada ciudad, ingreso de mayor a menor
v.sort_values(["ciudad", "ingreso"], ascending=[True, False]).head(8)

,fecha,producto,categoria,cantidad,precio_unitario,ciudad,ingreso,mes
94,2026-02-15,Sartén,Hogar,6,18.0,Cuenca,108.0,2
29,2026-01-15,Audífonos,Electrónica,4,25.0,Cuenca,100.0,1
107,2026-02-21,Mouse,Electrónica,6,15.0,Cuenca,90.0,2
136,2026-03-07,Audífonos,Electrónica,3,25.0,Cuenca,75.0,3
34,2026-01-18,Mouse,Electrónica,4,15.0,Cuenca,60.0,1
65,2026-02-03,Cargador,Electrónica,4,12.5,Cuenca,50.0,2
198,2026-03-31,Audífonos,Electrónica,2,25.0,Cuenca,50.0,3
153,2026-03-13,Licuadora,Hogar,1,45.0,Cuenca,45.0,3


### Su turno 1.1 · 3 min
Guarde en `top_quito` las **cinco** ventas de mayor ingreso en Quito, ordenadas de mayor a menor ingreso. Pista: filtre primero, ordene después (o use `nlargest`).

In [6]:
# TODO: reemplace los ... por su código
top_quito = v[v["ciudad"] == "Quito"].nlargest(5, "ingreso")
top_quito[["fecha", "producto", "cantidad", "ingreso"]]

,fecha,producto,cantidad,ingreso
188,2026-03-28,Audífonos,10,250.0
114,2026-02-23,Licuadora,5,225.0
53,2026-01-27,Audífonos,8,200.0
173,2026-03-21,Audífonos,6,150.0
23,2026-01-11,Sartén,8,144.0


In [7]:
assert len(top_quito) == 5, f"deben ser 5 filas y hay {len(top_quito)}"
assert (top_quito["ciudad"] == "Quito").all(), "hay ventas de otras ciudades: filtre antes de ordenar"
assert top_quito["ingreso"].iloc[0] == 250.0 and np.isclose(top_quito["ingreso"].sum(), 969.0), "revise el orden (ascending=False)"
print("Correcto: la venta más grande de Quito fueron 10 audífonos (250).")

Correcto: la venta más grande de Quito fueron 10 audífonos (250).


## 2. `groupby` con una clave: dividir, aplicar, combinar

`groupby` hace tres cosas en una línea: **divide** la tabla en grupos según una columna, **aplica** una función a cada grupo y **combina** los resultados en una tabla nueva, con una fila por grupo. El resultado es una Series (o DataFrame) cuyo **índice son los grupos**.

```
v.groupby("ciudad")["ingreso"].sum()
   └── clave ──┘ └ columna ┘ └ función ┘
```

In [8]:
v.groupby("ciudad")["ingreso"].sum()

ciudad
Cuenca        871.7
Guayaquil    2761.9
Machala       532.5
Manta         694.0
Quito        2564.7
Name: ingreso, dtype: float64

In [9]:
# El resultado es una Series: se ordena, se redondea, se grafica como cualquier otra
v.groupby("ciudad")["ingreso"].sum().sort_values(ascending=False).round(1)

ciudad
Guayaquil    2761.9
Quito        2564.7
Cuenca        871.7
Manta         694.0
Machala       532.5
Name: ingreso, dtype: float64

In [10]:
# Otras funciones: mean, median, min, max, count (valores no nulos), size (filas), nunique (valores distintos)
print("ticket promedio por ciudad:")
print(v.groupby("ciudad")["ingreso"].mean().round(2))
print()
print("ventas por ciudad (size) y productos distintos vendidos (nunique):")
print(v.groupby("ciudad").size())
print(v.groupby("ciudad")["producto"].nunique())

ticket promedio por ciudad:
ciudad
Cuenca       24.21
Guayaquil    36.34
Machala      44.38
Manta        43.38
Quito        42.74
Name: ingreso, dtype: float64

ventas por ciudad (size) y productos distintos vendidos (nunique):
ciudad
Cuenca       36
Guayaquil    76
Machala      12
Manta        16
Quito        60
dtype: int64
ciudad
Cuenca        9
Guayaquil    10
Machala       6
Manta         9
Quito        10
Name: producto, dtype: int64


`count()` cuenta valores **no nulos** de una columna; `size()` cuenta **filas** del grupo. Con datos limpios coinciden; con faltantes, no.

### Su turno 2.1 · 4 min
Guarde en `ticket_cat` el ingreso **promedio** por categoría (redondeado a 2 decimales) y en `ventas_cat` el **número de ventas** por categoría.

In [11]:
v

,fecha,producto,categoria,cantidad,precio_unitario,ciudad,ingreso,mes
0,2026-01-02,Sartén,Hogar,1,18.0,Quito,18.0,1
1,2026-01-02,Mouse,Electrónica,1,15.0,Machala,15.0,1
2,2026-01-03,Galletas,Alimentos,2,1.8,Quito,3.6,1
3,2026-01-03,Cuaderno,Papelería,2,2.5,Guayaquil,5.0,1
4,2026-01-04,Cargador,Electrónica,3,12.5,Guayaquil,37.5,1
...,...,...,...,...,...,...,...,...
195,2026-03-30,Cargador,Electrónica,8,12.5,Guayaquil,100.0,3
196,2026-03-30,Cargador,Electrónica,1,12.5,Quito,12.5,3
197,2026-03-31,Audífonos,Electrónica,2,25.0,Quito,50.0,3
198,2026-03-31,Audífonos,Electrónica,2,25.0,Cuenca,50.0,3


In [12]:
# TODO: reemplace los ... por su código
ticket_cat = v.groupby("categoria")["ingreso"].mean().round(2)
ventas_cat = v.groupby("categoria")["ingreso"].count()
print(ticket_cat)
print(ventas_cat)

categoria
Alimentos      11.81
Electrónica    53.09
Hogar          84.50
Papelería       5.88
Name: ingreso, dtype: float64
categoria
Alimentos      67
Electrónica    64
Hogar          36
Papelería      33
Name: ingreso, dtype: int64


In [13]:
assert np.isclose(ticket_cat["Electrónica"], 53.09) and np.isclose(ticket_cat["Alimentos"], 11.81), "ticket_cat: mean() del ingreso por categoría"
assert ventas_cat["Electrónica"] == 64 and ventas_cat.sum() == 200, "ventas_cat: size() cuenta filas por grupo"
print("Correcto. Fíjese: Alimentos tiene muchas ventas pequeñas; Hogar, pocas ventas grandes.")

Correcto. Fíjese: Alimentos tiene muchas ventas pequeñas; Hogar, pocas ventas grandes.


## 3. Varias claves y varias funciones: `agg`

Con una **lista de columnas** como clave, los grupos son todas las combinaciones (ciudad × categoría) y el resultado tiene un índice de dos niveles (`MultiIndex`). Con `agg` se piden varias funciones a la vez, y con la **agregación con nombre** cada resultado recibe el nombre de columna que usted elija.

In [14]:
por_cc = v.groupby(["ciudad", "categoria"])["ingreso"].sum()
por_cc.head(8)

ciudad     categoria  
Cuenca     Alimentos       107.4
           Electrónica     552.5
           Hogar           207.0
           Papelería         4.8
Guayaquil  Alimentos       284.1
           Electrónica     965.0
           Hogar          1413.0
           Papelería        99.8
Name: ingreso, dtype: float64

In [15]:
# unstack() lleva el segundo nivel del índice a columnas: una tabla ciudad × categoría
por_cc.unstack().fillna(0).round(1)

categoria,Alimentos,Electrónica,Hogar,Papelería
ciudad,,,,
Cuenca,107.4,552.5,207.0,4.8
Guayaquil,284.1,965.0,1413.0,99.8
Machala,117.0,30.0,360.0,25.5
Manta,106.0,505.0,63.0,20.0
Quito,176.7,1345.0,999.0,44.0


In [16]:
# Varias funciones sobre una columna
v.groupby("ciudad")["ingreso"].agg(["sum", "mean", "count"]).round(2)

,sum,mean,count
ciudad,,,
Cuenca,871.7,24.21,36
Guayaquil,2761.9,36.34,76
Machala,532.5,44.38,12
Manta,694.0,43.38,16
Quito,2564.7,42.74,60


In [17]:
# Agregación con nombre: nombre_nuevo = (columna, función). Es la forma más legible.
resumen = v.groupby("ciudad").agg(
    ingreso=("ingreso", "sum"),
    ticket=("ingreso", "mean"),
    ventas=("ingreso", "size"),
    unidades=("cantidad", "sum"),
).round(2)
resumen

,ingreso,ticket,ventas,unidades
ciudad,,,,
Cuenca,871.7,24.21,36,92
Guayaquil,2761.9,36.34,76,256
Machala,532.5,44.38,12,45
Manta,694.0,43.38,16,58
Quito,2564.7,42.74,60,184


In [18]:
# El resultado de groupby tiene los grupos en el ÍNDICE. reset_index() los convierte en una columna normal.
resumen.reset_index().head(3)

,ciudad,ingreso,ticket,ventas,unidades
0,Cuenca,871.7,24.21,36,92
1,Guayaquil,2761.9,36.34,76,256
2,Machala,532.5,44.38,12,45


### Su turno 3.1 · 5 min
Construya `por_producto`: una tabla con una fila por **producto** y las columnas `unidades` (suma de cantidad), `ingreso` (suma) y `ventas` (número de filas), ordenada por `unidades` de mayor a menor. Use agregación con nombre.

In [19]:
# TODO: reemplace los ... por su código
por_producto = v.groupby("producto").agg(
    unidades=("cantidad", "sum"),
    ingreso=("ingreso", "sum"),
    ventas=("ingreso", "size")
).sort_values("unidades", ascending=False)
por_producto

,unidades,ingreso,ventas
producto,,,
Cargador,75,937.5,24
Galletas,73,131.4,24
Café,70,455.0,22
Sartén,69,1242.0,21
Audífonos,66,1650.0,18
Chocolate,64,204.8,21
Cuaderno,63,157.5,15
Bolígrafo,61,36.6,18
Mouse,54,810.0,22


In [20]:
assert list(por_producto.columns) == ["unidades", "ingreso", "ventas"], f"columnas: {list(por_producto.columns)}"
assert len(por_producto) == 10 and por_producto["ventas"].sum() == 200
assert por_producto.index[0] == "Cargador" and por_producto.loc["Cargador", "unidades"] == 75, "ordene por unidades, de mayor a menor"
assert np.isclose(por_producto.loc["Licuadora", "ingreso"], 1800.0)
print("Correcto: el cargador es el más vendido en unidades (75), pero la licuadora genera más ingreso.")

Correcto: el cargador es el más vendido en unidades (75), pero la licuadora genera más ingreso.


## 4. De la agregación al indicador (KPI)

Los KPIs de la sesión 3 eran frases; hoy son una línea de código cada uno:

| KPI (sesión 3) | Código |
| :-- | :-- |
| Ingreso total | `v["ingreso"].sum()` |
| Ticket promedio | `v["ingreso"].mean()` |
| Participación de cada ciudad en el ingreso (%) | `v.groupby("ciudad")["ingreso"].sum() / v["ingreso"].sum() * 100` |
| Unidades por venta en cada categoría | `v.groupby("categoria")["cantidad"].mean()` |
| Producto más vendido por ciudad | `v.groupby(["ciudad", "producto"])["cantidad"].sum()` + ordenar + `groupby("ciudad").head(1)` |

Dos herramientas nuevas aparecen aquí: **dividir una Series agregada por un total** (porcentajes) y **`transform`**, que devuelve el resultado del grupo **repetido en cada fila**, para comparar cada venta con su grupo.

In [21]:
# Participación de cada ciudad (%)
(v.groupby("ciudad")["ingreso"].sum() / v["ingreso"].sum() * 100).round(1).sort_values(ascending=False)

ciudad
Guayaquil    37.2
Quito        34.5
Cuenca       11.7
Manta         9.3
Machala       7.2
Name: ingreso, dtype: float64

In [22]:
# transform: el promedio de SU ciudad al lado de cada venta -> ¿esta venta está por encima del promedio de su ciudad?
v["ticket_ciudad"] = v.groupby("ciudad")["ingreso"].transform("mean")
v["sobre_promedio"] = v["ingreso"] > v["ticket_ciudad"]
v[["ciudad", "producto", "ingreso", "ticket_ciudad", "sobre_promedio"]].head(6)

,ciudad,producto,ingreso,ticket_ciudad,sobre_promedio
0,Quito,Sartén,18.0,42.745000,False
1,Machala,Mouse,15.0,44.375000,False
2,Quito,Galletas,3.6,42.745000,False
3,Guayaquil,Cuaderno,5.0,36.340789,False
4,Guayaquil,Cargador,37.5,36.340789,True
5,Guayaquil,Cuaderno,10.0,36.340789,False


In [23]:
# Producto más vendido (en unidades) en cada ciudad: agrupar por dos claves, ordenar y quedarse con la primera fila de cada ciudad
unidades = v.groupby(["ciudad", "producto"])["cantidad"].sum().reset_index()
unidades.sort_values(["ciudad", "cantidad"], ascending=[True, False]).groupby("ciudad").head(1)

,ciudad,producto,cantidad
5,Cuenca,Galletas,24
12,Guayaquil,Cargador,44
20,Machala,Café,18
32,Manta,Mouse,17
34,Quito,Audífonos,38


### Su turno 4.1 · 4 min
Guarde en `pct_ciudad` la participación (%) de cada ciudad en el ingreso total, redondeada a 1 decimal y ordenada de mayor a menor. La suma debe dar 100.

In [24]:
# TODO: reemplace los ... por su código
pct_ciudad = (v.groupby("ciudad")["ingreso"].sum() / v["ingreso"].sum() * 100).sort_values(ascending=False).round(1)
pct_ciudad

ciudad
Guayaquil    37.2
Quito        34.5
Cuenca       11.7
Manta         9.3
Machala       7.2
Name: ingreso, dtype: float64

In [25]:
assert np.isclose(pct_ciudad["Guayaquil"], 37.2) and np.isclose(pct_ciudad["Quito"], 34.5), "divida la suma por ciudad entre el total y multiplique por 100"
assert np.isclose(pct_ciudad.sum(), 100, atol=0.2) and pct_ciudad.index[0] == "Guayaquil"
print("Correcto: Guayaquil y Quito concentran", round(pct_ciudad["Guayaquil"] + pct_ciudad["Quito"], 1), "% del ingreso.")

Correcto: Guayaquil y Quito concentran 71.7 % del ingreso.


## 5. `pivot_table`: la tabla dinámica

`pivot_table` es `groupby` con dos claves presentado como una **tabla de doble entrada**: una clave en las filas (`index`), otra en las columnas (`columns`), una columna de valores (`values`) y la función de resumen (`aggfunc`). `fill_value` reemplaza las combinaciones sin datos y `margins=True` añade los totales.

In [26]:
v.pivot_table(index="ciudad", columns="mes", values="ingreso", aggfunc="sum", fill_value=0, margins=True).round(1)

mes,1,2,3,All
ciudad,,,,
Cuenca,195.0,362.1,314.6,871.7
Guayaquil,940.1,947.7,874.1,2761.9
Machala,56.0,209.0,267.5,532.5
Manta,59.6,99.6,534.8,694.0
Quito,585.1,781.7,1197.9,2564.7
All,1835.8,2400.1,3188.9,7424.8


In [27]:
# Lo mismo con groupby + unstack (sin totales). pivot_table es el atajo.
v.groupby(["ciudad", "mes"])["ingreso"].sum().unstack(fill_value=0).round(1)

mes,1,2,3
ciudad,,,
Cuenca,195.0,362.1,314.6
Guayaquil,940.1,947.7,874.1
Machala,56.0,209.0,267.5
Manta,59.6,99.6,534.8
Quito,585.1,781.7,1197.9


In [28]:
# aggfunc puede ser otra función o varias
v.pivot_table(index="categoria", columns="ciudad", values="ingreso", aggfunc="mean").round(1)

ciudad,Cuenca,Guayaquil,Machala,Manta,Quito
categoria,,,,,
Alimentos,7.7,10.9,29.2,17.7,10.4
Electrónica,42.5,46.0,15.0,72.1,64.0
Hogar,51.8,108.7,120.0,31.5,71.4
Papelería,1.0,6.2,8.5,20.0,5.5


### Su turno 5.1 · 4 min
Cree `unidades_mes`: una tabla dinámica con los **productos** en las filas, los **meses** en las columnas, la **suma de `cantidad`** en las celdas, ceros donde no haya datos y totales (`margins=True`).

In [29]:
# TODO: reemplace los ... por su código
unidades_mes = v.pivot_table(index="producto", columns="mes", values="cantidad", aggfunc="sum", margins=True)
unidades_mes

mes,1,2,3,All
producto,,,,
Audífonos,17,10,39,66
Bolígrafo,17,16,28,61
Café,18,23,29,70
Cargador,7,39,29,75
Chocolate,22,21,21,64
Cuaderno,19,11,33,63
Galletas,9,26,38,73
Licuadora,9,17,14,40
Mouse,15,23,16,54


In [30]:
assert list(unidades_mes.shape) == [11, 4], f"forma {unidades_mes.shape}: 10 productos + total, 3 meses + total"
assert unidades_mes.loc["Cargador", 1] == 7 and unidades_mes.loc["Café", "All"] == 70
assert unidades_mes.loc["All", 3] == 278, "la fila All debe sumar las unidades de marzo"
print("Correcto: marzo fue el mes con más unidades (", unidades_mes.loc["All", 3], ").")

Correcto: marzo fue el mes con más unidades ( 278 ).


## 6. `crosstab`: contar combinaciones

`pd.crosstab(filas, columnas)` **cuenta** cuántas filas hay en cada combinación de dos columnas categóricas. Con `normalize="index"` cada fila suma 1 (proporción por fila); con `normalize="columns"`, cada columna suma 1. Es la herramienta para preguntas del tipo «¿qué proporción de las ventas de cada ciudad es electrónica?».

In [31]:
pd.crosstab(v["ciudad"], v["categoria"], margins=True)

categoria,Alimentos,Electrónica,Hogar,Papelería,All
ciudad,,,,,
Cuenca,14,13,4,5,36
Guayaquil,26,21,13,16,76
Machala,4,2,3,3,12
Manta,6,7,2,1,16
Quito,17,21,14,8,60
All,67,64,36,33,200


In [37]:
# Proporción dentro de cada ciudad (cada fila suma 1)
pd.crosstab(v["ciudad"], v["categoria"], normalize="index").round(3)

categoria,Alimentos,Electrónica,Hogar,Papelería
ciudad,,,,
Cuenca,0.389,0.361,0.111,0.139
Guayaquil,0.342,0.276,0.171,0.211
Machala,0.333,0.167,0.250,0.250
Manta,0.375,0.438,0.125,0.062
Quito,0.283,0.350,0.233,0.133


In [33]:
# crosstab también agrega otra columna si se le pasan values y aggfunc (entonces es igual a pivot_table)
pd.crosstab(v["ciudad"], v["categoria"], values=v["ingreso"], aggfunc="sum").round(1)

categoria,Alimentos,Electrónica,Hogar,Papelería
ciudad,,,,
Cuenca,107.4,552.5,207.0,4.8
Guayaquil,284.1,965.0,1413.0,99.8
Machala,117.0,30.0,360.0,25.5
Manta,106.0,505.0,63.0,20.0
Quito,176.7,1345.0,999.0,44.0


### Su turno 6.1 · 3 min
Guarde en `mezcla` la proporción de ventas de cada categoría **dentro de cada ciudad** (filas = ciudad, columnas = categoría, cada fila suma 1), redondeada a 3 decimales. ¿En qué ciudad pesa más la electrónica?

In [41]:
# TODO: reemplace los ... por su código
mezcla = pd.crosstab(v["ciudad"], v["categoria"], normalize="index").round(3)
mezcla

categoria,Alimentos,Electrónica,Hogar,Papelería
ciudad,,,,
Cuenca,0.389,0.361,0.111,0.139
Guayaquil,0.342,0.276,0.171,0.211
Machala,0.333,0.167,0.250,0.250
Manta,0.375,0.438,0.125,0.062
Quito,0.283,0.350,0.233,0.133


In [42]:
assert np.isclose(mezcla.loc["Guayaquil", "Electrónica"], 0.276) and np.isclose(mezcla.loc["Manta", "Hogar"], 0.125), "normalize='index' (cada fila suma 1)"
assert np.allclose(mezcla.sum(axis=1), 1, atol=0.01)
print("Correcto. La electrónica pesa más en", mezcla["Electrónica"].idxmax(), ".")

Correcto. La electrónica pesa más en Manta .


## 7. Errores frecuentes

| Síntoma | Causa | Solución |
| :-- | :-- | :-- |
| `KeyError: 'ciudad'` después de un `groupby` | la clave quedó en el índice, no en las columnas | `reset_index()` o `as_index=False` |
| el promedio de los promedios no coincide con el promedio general | los grupos tienen tamaños distintos | agrupe sobre la tabla original, no sobre promedios |
| `count` da menos que `size` | hay valores faltantes en la columna | decida en la limpieza; use `size` para contar filas |
| la tabla dinámica tiene `NaN` | combinaciones sin datos | `fill_value=0` (si cero es la respuesta correcta) |
| `groupby` con una columna `category` devuelve grupos vacíos | categorías sin datos | `groupby(..., observed=True)` |

## 8. Hacia el proyecto: de la pregunta al `groupby`

Las **tres preguntas de negocio** de su ficha y los **dos KPIs** de la Tarea 1 son operaciones de este notebook. En el proyecto tendrá que responderlas con agregaciones y gráficos: empiece hoy.

### Su turno 8.1 · en casa
Con el conjunto de datos de su proyecto (ya cargado y limpio como en la Tarea 1): (a) responda **una** de sus preguntas de negocio con `groupby` + `agg`; (b) construya **una** `pivot_table` o `crosstab` que compare dos variables categóricas; (c) escriba en una celda de texto qué dice cada tabla en una frase. Traiga el notebook a la S9.

In [ ]:
# Plantilla para su proyecto (complete con sus columnas)
# datos = pd.read_csv("ruta/a/su_dataset.csv")
# datos.groupby("columna_categorica").agg(indicador=("columna_numerica", "sum"))
# pd.crosstab(datos["categoria_1"], datos["categoria_2"], normalize="index").round(2)

### Reto (opcional) · ¿Qué ciudad tuvo el ticket promedio más alto en cada mes?
Agrupe por `mes` y `ciudad`, calcule el ingreso promedio y, para cada mes, encuentre la ciudad con el valor máximo (`idxmax` por nivel, o `sort_values` + `groupby("mes").head(1)`).

In [ ]:
# TODO: reemplace los ... por su código
# Opcional: el resultado debe ser una Series con índice mes (1, 2, 3) y la ciudad como valor
mejor_ticket = ...
mejor_ticket

**Antes de la próxima sesión (S9):** entregue la **Tarea 1** al inicio de la S9 (un notebook por grupo, por el aula virtual) y haga la práctica `S8_Practica_Limpieza_Groupby.ipynb` (sin nota). Lecturas: McKinney cap. 10 §10.1–10.2 y 10.5; VanderPlas «Aggregation and Grouping» y «Pivot Tables».